In [34]:
from typing_extensions import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from typing import Annotated, List
import operator
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_community.utilities import GoogleSerperAPIWrapper
import pprint
import json


In [35]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["SERPER_API_KEY"] = os.getenv("SERPER_API_KEY")


In [36]:
llm = ChatGroq(model="gemma2-9b-it")

In [339]:
class Section(BaseModel):
    title: str = Field(
        description="A title of the news",
    )
    link: str = Field(
        description="A link to the news",
    )
    time_published: str = Field(
        description="A time when news published",
    )
    source: str = Field(
        description="A news chennel name",
    )
    # snippet: str = Field(
    #     description="News overview",
    # )

class Sections(BaseModel):
    sections: List[Section] = Field(
        description="Sections of the AI news",
    )

class query(BaseModel):
    news_topic: str = Field(description="string for topic")


class NewsItem(BaseModel):
    title: str
    link: str
    date: str
    source: str

class RelevantNews(BaseModel):
    results: List[NewsItem]

relevant_news_agent = llm.with_structured_output(RelevantNews)

# Augment the LLM with schema for structured output
planner = llm.with_structured_output(Sections)
newsai = llm.with_structured_output(query)


In [278]:
orchestration_instruction = """
# Orchestrator Agent Instructions

**Role:**  
You are the Orchestrator Agent in a multi-agent AI system for curating and summarizing AI-related news.

**Objective:**  
Analyze a list of news items and generate a **structured outline (plan)** for a news report.

**Input Format:**  
You will receive the latest AI news as a list of dictionaries. Each news item includes:
- **title**
- **link**
- **date** (time published)
- **source**

**Tasks:**  
1. **Analyze:** Carefully review each news item.
2. **Organize:** Arrange the news into a clear, well-structured outline. Group items by category or relevance if possible.
3. **Output:** Produce a well-organized outline (plan) based solely on the provided news information.

Return only the **structured outline** as your final output.
"""

worker_instruction = """
# Worker Agent Instructions

**Role:**  
You are a Worker Agent in news summarization system tasked with transforming a single news item into an engaging and insightful summary. make sure heading should align properly.

**Steps to Follow:**

2. **Title:**  
   - Provide a compelling and relevant title for the news.

3. **Source & Timestamp:**  
   - Immediately below the title, include the news source and publication time.

4. **Article Understanding:**  
   - Use the provided link to read and fully understand the article.

5. **Snippet:**  
   - Based on article understanding, Write a brief, engaging and factual snippet that captures the essence of the news.

6. **Summary:**  
   - Offer a concise 3-5 sentence summary that covers the key points.

7. **Analysis:**  
   - Provide a short analysis discussing the article's significance, its relation to past developments, and its potential future impact.

8. **Key Insights:**  
   - List 2-3 bullet points highlighting the most important takeaways.

9. **Original Source:**  
   - Conclude with the direct link to the original article.

**Formatting Requirements:**  
- Use clean **Markdown formatting** with clear headings and highlights.
- Do not include any additional commentary or extraneous text.
"""

news_instruction = """
You are an intelligent news-search agent. Your role is to deeply understand the user's query and generate a concise, information-rich search topic that can retrieve all relevant news articles from google serper.
below is the code where we will pass topic. topic should be string so you just have to give a small string.

def news(topic: str):
    search = GoogleSerperAPIWrapper(type="news", tbs="qdr:h24")
    results = search.results(topic)

"""

verification_instruction = """
You are an AI assistant named VerifiNews. Your role is to carefully examine each news dictionary in a list passed to you.

Each dictionary has the fields: `title`, `link`, `date`, and `source`.

Keep only those dictionaries that:
- Have a complete and meaningful `title`
- Have a valid `link` 
- Have a `date` in correct format 
- Have a valid `source`

Remove any news dictionary that is:
- duplicate
- Incomplete
- Has junk or empty values
- Has a broken or invalid URL
- Has an incorrect or missing date

Return only the cleaned list of valid news dictionaries. Do not return anything else.
"""

analysis_instruction = """
You are a knowledge analyst agent with access to the web.

TASK:
Given the title, link, publication date, and source of a news article, your job is to:

1. **Fetch the article content** from the provided URL.3. Immediately below the title, include the news source and publication time.

2. Conduct an **in-depth factual analysis**, including:
   - Summary of the article (clear, concise, accurate)
   - Business or technological insights (if applicable)
   - Psychological, societal, or philosophical implications (if relevant)
   - Future impact or long-term significance
3. Identify the article's **relevance** to one or more of these domains:
   - Artificial Intelligence (AI)
   - Environment and Climate
   - Economics and Investing
   - Ethics, Philosophy, or Psychology
4. Provide a well-structured final output with the following sections:
   - Summary
   - Key Insights
   - Broader Implications
   - Future Impact
   - Relevance
   - Conclusion

5. **Original Source:**  
   - Conclude with the direct link to the original article.

RULES:
- Stick to **factual information** from the article and reliable sources.
- If the link is broken or the article can't be fetched, report that clearly.
- Do not fabricate content.
- Use markdown for formatting.

INPUT:
- Title: {title}
- Link: {link}
- Date: {date}
- Source: {source}
"""

explainer_instructions = """
You are Explainer, an intelligent helpful assistant.
Answer the users question acurately in short.
Provide factual answers.
"""

In [295]:
import feedparser
from datetime import datetime, timedelta, timezone
import requests

# RSS feeds organized by type
rss_feeds = {
    "Top Stories": [
        "https://www.cbc.ca/cmlink/rss-topstories",
        "https://www.thestar.com/search/?f=rss&t=article&bl=2827101&l=20",
        "https://globalnews.ca/feed/",
        "https://nationalpost.com/feed/",
    ],
    "World News": [
        "https://www.cbc.ca/cmlink/rss-world",
        "https://globalnews.ca/world/feed/",
        "http://rss.cnn.com/rss/edition_world.rss"
    ],
    "Canada News": [
        "https://www.cbc.ca/cmlink/rss-canada",
        "https://globalnews.ca/canada/feed/"
    ],
    "Business": [
        "https://www.cbc.ca/cmlink/rss-business",
        "https://www.theglobeandmail.com/report-on-business/?service=rss",
        "https://business.financialpost.com/feed/"
    ],
    "Technology": [
        "https://www.cbc.ca/cmlink/rss-technology",
        "https://www.theglobeandmail.com/technology/?service=rss"
    ],
    "Politics": [
        "https://globalnews.ca/politics/feed/",
        "https://www.cbc.ca/webfeed/rss/rss-politics"
    ],
    "Health": [
        "https://globalnews.ca/health/feed/",
        "https://www.cbc.ca/webfeed/rss/rss-health"
    ]
}

# Time setup for 24-hour filtering
now = datetime.now(timezone.utc)
print(now)
print(timedelta(days=1))
last_24hrs = now - timedelta(days=1)
print(last_24hrs)

# Final result list
all_news = []
print(rss_feeds.items())


2025-03-31 09:27:58.110360+00:00
1 day, 0:00:00
2025-03-30 09:27:58.110360+00:00
dict_items([('Top Stories', ['https://www.cbc.ca/cmlink/rss-topstories', 'https://www.thestar.com/search/?f=rss&t=article&bl=2827101&l=20', 'https://globalnews.ca/feed/', 'https://nationalpost.com/feed/']), ('World News', ['https://www.cbc.ca/cmlink/rss-world', 'https://globalnews.ca/world/feed/', 'http://rss.cnn.com/rss/edition_world.rss']), ('Canada News', ['https://www.cbc.ca/cmlink/rss-canada', 'https://globalnews.ca/canada/feed/']), ('Business', ['https://www.cbc.ca/cmlink/rss-business', 'https://www.theglobeandmail.com/report-on-business/?service=rss', 'https://business.financialpost.com/feed/']), ('Technology', ['https://www.cbc.ca/cmlink/rss-technology', 'https://www.theglobeandmail.com/technology/?service=rss']), ('Politics', ['https://globalnews.ca/politics/feed/', 'https://www.cbc.ca/webfeed/rss/rss-politics']), ('Health', ['https://globalnews.ca/health/feed/', 'https://www.cbc.ca/webfeed/rss/rs

In [296]:
last_24hrs

datetime.datetime(2025, 3, 30, 9, 27, 58, 110360, tzinfo=datetime.timezone.utc)

checkout difference b/w times.

In [41]:
# fetch_rss.py (with timeout + fallback)
def rss_news():
    rss_feeds_news = []
    for urls in rss_feeds.values():
            for url in urls:
                try:
                    # Fetch RSS with timeout
                    response = requests.get(url, timeout=5)
                    feed = feedparser.parse(response.content)

                    for entry in feed.entries:
                        # print(entry)
                        try:
                            published_time = datetime(*entry.published_parsed[:6])
                            published_time = published_time.replace(tzinfo=timezone.utc)
                            print("published_time", published_time)
                            if published_time > last_24hrs:
                                # print(entry.title)
                                news_item = {
                                    "title": entry.title,
                                    "link": entry.link,
                                    "date": published_time.strftime('%Y-%m-%d %H:%M:%S'), #todo
                                    "source": feed.feed.get("title", "Unknown Source"), #todo
                                }
                                rss_feeds_news.append(news_item)
                        except Exception as e:
                            print(f"⚠️ Entry error: {e}")
                            continue

                except requests.exceptions.Timeout:
                    print(f"⏰ Timeout: {url}")
                except Exception as e:
                    print(f"❌ Failed to fetch {url}: {e}")

    return rss_feeds_news


DB

In [297]:
# db_handler.py
import sqlite3
from datetime import datetime, timedelta, timezone

DB_NAME = "news_ai.db"

In [ ]:
# Initialize database and table
def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS news (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            link TEXT UNIQUE,
            date TEXT,
            source TEXT,
            fetched_at       
        )
    ''')
    conn.commit()
    conn.close()

# Insert news into database (skip duplicates)
def insert_news(news_list):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    print("insert start")
    for news in news_list:
        try:
            cursor.execute('''
                INSERT OR IGNORE INTO news (title, link, date, source, fetched_at)
                VALUES (?, ?, ?, ?, ?)
            ''', (news["title"], news["link"], news["date"], news["source"], now))
        except Exception:
            continue
    print("insert done")
    conn.commit()
    conn.close()


# Delete news older than X hours (default: 48 hours)
def delete_old_news(hours=48):
    threshold_time = (datetime.now(timezone.utc) - timedelta(hours=hours)).strftime('%Y-%m-%d %H:%M:%S')
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM news WHERE fetched_at < ?", (threshold_time,))
    conn.commit()
    conn.close()


# Get news by optional filters (e.g., keyword only)
def get_rss_news(keyword=None, limit=100):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    print(keyword)
    query = "SELECT title, link, date, source FROM news WHERE 1=1"
    params = []

    if keyword:
        query += " AND LOWER(title) LIKE ?"
        params.append(f"%{keyword.lower()}%")

    query += " ORDER BY date DESC LIMIT ?"
    params.append(limit)

    cursor.execute(query, params)
    results = cursor.fetchall()
    conn.close()

    return [
        {"title": r[0], "link": r[1], "date": r[2], "source": r[3]}
        for r in results
    ]

def chunk_news(news_list, size=40):
    return [news_list[i:i+size] for i in range(0, len(news_list), size)]


# def get_rss_news(news_list, user_query):
#     relevant_chunks = []

#     for chunk in chunk_news(news_list):
#         prompt = f"""
#         User asked: "{user_query}"

#         Here is a list of news articles (title, date, source, link):

#         {json.dumps(chunk, indent=2)}

#         Please extract only the articles that are relevant to the user's query.
#         Return them as a JSON list of dictionaries.
#         """
#         result = llm.predict(prompt)
#         filtered = json.loads(result)
#         relevant_chunks.extend(filtered)

In [ ]:
news_list

In [350]:
size = 40
chunk = [news_list[i:i+size] for i in range(0, len(news_list), size)]

In [ ]:
chunk

In [ ]:
user_query = "Pierre Poilievre"
relevant_chunks = []
for chunk in chunk:
        prompt = f"""
        User asked: "{user_query}"

        Here is a list of news articles (title, date, source, link):

        {json.dumps(chunk, indent=2)}

        Please extract only the articles that are relevant to the user's query.
        Return them as a JSON list of dictionaries.
        """
        result = relevant_news_agent.invoke(prompt)
        print(result.results)
        cleaned = [item.model_dump() for item in result.results]

        
        relevant_chunks.extend(cleaned)

relevant_chunks

In [299]:
init_db()

In [ ]:
news_list = rss_news()

In [ ]:
news_list

In [302]:
insert_news(news_list=news_list)

insert start
insert done


In [303]:
delete_old_news()

In [308]:
from langgraph.constants import Send
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ExplainerState(TypedDict):
    question: str
    answer: str

# Graph state
class State(TypedDict):
    news_topic: str  # news topic
    latest_news: list[dict]
    sections: list[Section]  # List of news sections
    completed_sections: Annotated[list, operator.add]  # All workers write to this key in parallel
    final_report: str  # Final report

# Worker state
class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]

# Nodes
def google_news(topic: str):
    search = GoogleSerperAPIWrapper(type="news", tbs="qdr:h24")
    results = search.results(topic)
    #pprint.pp(results)
    
    for news in results["news"]:
        for key in ['imageUrl', 'position', 'snippet']:
            if key in news:
                del news[key] 

    #pprint.pp(results)
    return results['news'] 

def news(topic: str):
    g_news = google_news(topic)
    r_news = get_rss_news(keyword=topic)
    print("r_news = ",r_news)
    latest_news = g_news + r_news
    return latest_news

def news_ai(state: State):
    """AI agent that understand user query and generate string for news extraction"""

    news_topic = newsai.invoke(
        [
            SystemMessage(content=news_instruction),
            HumanMessage(
                content=f"Here is the user query: {state['news_topic']}"
            ),
        ]
    )
    #news_topic_dict = news_topic.model_dump()
    # print("news_topic:", news_topic_dict)

    topic = news_topic.news_topic
    print(topic)
    # g_news = google_news(topic)
    # r_news = get_rss_news(keyword=topic)
    # latest_news = g_news + r_news
    latest_news = news(topic=topic)
    print("latest =", latest_news)
    return {"latest_news": latest_news}
    
def verify_news(state: State):
    verified_news = llm.invoke(
        [
            SystemMessage(content=verification_instruction),
            HumanMessage(
                content=f"Here is all the news details: {state['latest_news']}"
            ),
        ]
    )
    print("verified = ", verified_news)
    return {"latest_news": verified_news}


def orchestrator(state: State):
    """Orchestrator that generates a plan for the news"""

    #print("Orchestrator is activated, here output will be lst of section: State")
    # latest_news = news(state["news_topic"])
    # print(latest_news)
    # Generate queries
    report_sections = planner.invoke(
        [
            SystemMessage(content=orchestration_instruction),
            HumanMessage(
                content=f"Here is the latest AI news: {state['latest_news']}"
            ),
        ]
    )
    #print(state["topic"])
    print("Report Sections:",report_sections)

    return {"sections": report_sections.sections}

def llm_call(state: WorkerState):
    """Worker writes a section of the report"""

    #print("llm_call is activated, here output will be lst of section: WorkerState, section = ")
    # print(f"state[section] = {state['section']}")
    # Generate section
    section = llm.invoke(
        [
            SystemMessage(content=analysis_instruction),
            HumanMessage(
                content=f"Here is the news_title: {state['section'].title},\
                      news_link: {state['section'].link}, time_published: {state['section'].time_published},\
                          news_source: {state['section'].source}" # news_snippet: {state['section'].snippet}
            ),
        ]
    )

    #print(f"{state['section']},\n {type(state['section'])}")
    # Write the updated section to completed sections
    #print("section = \n", section)
    return {"completed_sections": [section.content]}


def synthesizer(state: State):
    """Synthesize full report from sections"""

    #print("synthesizer is activated, here output will be lst of completed sections: State")

    # List of completed sections
    completed_sections = state["completed_sections"]
    #print("completed_sections",completed_sections)
    # Format completed section to str to use as context for final sections
    completed_report_sections = "\n\n---\n\n".join(completed_sections)

    #print("completed_report_sections",completed_report_sections)

    return {"final_report": completed_report_sections}


# Conditional edge function to create llm_call workers that each write a section of the report
def assign_workers(state: State):
    """Assign a worker to each section in the plan"""
    #print([s for s in state["sections"]])
    # Kick off section writing in parallel via Send() API
    return [Send("llm_call", {"section": s}) for s in state["sections"]]

def explainer(state: ExplainerState):
    
    # messages = state["messages"]

    # question = messages[0].content
    explainer = llm
    response = explainer.invoke(
        [
            SystemMessage(content=explainer_instructions),
            HumanMessage(
                content=f"Here is the question: {state['question']}"
            ),
        ]
    )
   
    # print(state["question"])
    # print("response: ", response)
 
    return {"answer": response.content}

In [ ]:
news("israel")

In [309]:
# Build workflow
builder = StateGraph(State)
builder_2 = StateGraph(ExplainerState)

# Add the nodes
builder.add_node("news_ai", news_ai)
# builder.add_node("verify_news", verify_news)
builder.add_node("orchestrator", orchestrator)
builder.add_node("llm_call", llm_call)
builder.add_node("synthesizer", synthesizer)
builder_2.add_node("explainer", explainer)

# Add edges to connect nodes
builder.add_edge(START, "news_ai")
builder.add_edge("news_ai", "orchestrator")
# builder.add_edge("news_ai", "verify_news")
# builder.add_edge("verify_news", "orchestrator")
builder.add_conditional_edges(
    "orchestrator", assign_workers, ["llm_call"]
)
builder.add_edge("llm_call", "synthesizer")
builder.add_edge("synthesizer", END)

# new graph..
builder_2.add_edge(START, "explainer")
builder_2.add_edge("explainer", END)

# Compile the workflow
graph = builder.compile()
graph_2 = builder_2.compile()

graph, graph_2

(<langgraph.graph.state.CompiledStateGraph at 0x220d5fdace0>,
 <langgraph.graph.state.CompiledStateGraph at 0x220d5fd9de0>)

In [310]:
# Invoke
response = graph.invoke({"news_topic": "Pierre Poilievre"})

Pierre Poilievre
Pierre Poilievre
r_news =  [{'title': 'Pierre Poilievre touts capital gains tax break in wake of reports of internal party tension', 'link': 'https://www.thestar.com/politics/federal-elections/pierre-poilievre-touts-capital-gains-tax-break-in-wake-of-reports-of-internal-party-tension/article_e0b4175f-b555-4f38-bbbb-be79b044d71f.html', 'date': '2025-03-30 23:53:00', 'source': 'www.thestar.com - RSS Results of type article'}, {'title': 'Pierre Poilievre unveils capital gains tax break ahead of election', 'link': 'https://globalnews.ca/news/11105333/conservative-party-leader-capital-gains-tax-break/', 'date': '2025-03-30 18:55:57', 'source': ''}, {'title': "Is Pierre Poilievre 'too Trumpy?' Does Mark Carney know his candidates' names? Five moments that shook up the election campaign this week", 'link': 'https://www.thestar.com/politics/federal-elections/is-pierre-poilievre-too-trumpy-does-mark-carney-know-his-candidates-names-five-moments-that/article_9d2b95fb-1d25-4746-b

In [283]:
response

{'news_topic': 'todays top tech news',
 'latest_news': [{'title': 'Chinese tech firms poach talent from Microsoft and Intel in Taiwan',
   'link': 'https://cybernews.com/news/china-workers-taiwan-microsoft-intel/',
   'date': '35 minutes ago',
   'source': 'Cybernews'},
  {'title': 'High Growth Tech Stocks in Europe for March 2025',
   'link': 'https://finance.yahoo.com/news/high-growth-tech-stocks-europe-050808770.html',
   'date': '4 hours ago',
   'source': 'Yahoo Finance'},
  {'title': '4 Top Tech Stocks to Buy Right Now',
   'link': 'https://www.msn.com/en-us/money/companies/4-top-tech-stocks-to-buy-right-now/ar-AA1BVXDv',
   'date': '10 hours ago',
   'source': 'MSN'},
  {'title': 'Elliott Wave analysis: S&P 500, Nasdaq 100 and top tech stocks [Video]',
   'link': 'https://www.fxstreet.com/news/elliott-wave-analysis-sp-500-nasdaq-100-and-top-tech-stocks-video-202503310121',
   'date': '8 hours ago',
   'source': 'FXStreet'},
  {'title': "Charles Barkley reacts to Florida comeback

In [311]:
from IPython.display import Markdown
Markdown(response["final_report"])

##  Economy and Especially Trump: Canadians' Thoughts on Campaigns

**Source:** NonStop Local KHQ  
**Publication Time:** 8 hours ago

### Summary

This article explores the opinions of Canadians regarding the ongoing US presidential campaign, focusing particularly on the impact of the economy and former President Donald Trump's involvement. Through interviews with Canadians, the article highlights concerns about the potential consequences of the US election on the Canadian economy, particularly in sectors like manufacturing and trade. The article also touches upon Canadians' mixed views on Donald Trump, with some expressing worries about his policies and others acknowledging his business acumen.  

### Key Insights

* **Economic Anxiety:** Canadians express concerns about the potential impact of a volatile US economy on their own financial stability, particularly in light of rising inflation and supply chain disruptions. 
* **Trump's Influence:** Despite living in a separate country, Canadians acknowledge the significant influence of US politics on their lives.  There is a mixed perception of Trump, with some viewing him as a strong leader while others criticize his policies and rhetoric. 
* **Cross-Border Ties:** The article emphasizes the strong economic and social ties between the US and Canada, highlighting the interconnected nature of their economies and the potential for spillover effects from US political events.

### Broader Implications

This article sheds light on the global impact of US elections, demonstrating how events in one nation can significantly influence the lives of citizens in other countries. It also reveals the anxieties surrounding economic uncertainty and the diverse opinions individuals hold about political figures and their potential impact.

### Future Impact

The outcome of the US election will likely continue to shape Canadians' perceptions of the economic and political landscape. Depending on the result, Canadians may experience varying levels of economic stability and uncertainty, potentially influencing their political engagement and views on cross-border relations.

### Relevance

* **Economics and Investing:** The article directly addresses the potential economic impacts of US political events on Canadians, making it relevant to understanding the global economic landscape and investment strategies.
* **Ethics, Philosophy, or Psychology:** The article touches upon the psychological impact of political uncertainty on individuals and explores the diverse perspectives Canadians hold on a prominent political figure, highlighting the complexities of individual and societal responses to political events.

### Conclusion

The article provides a glimpse into the concerns and perspectives of Canadians regarding the US presidential campaign. It underscores the interconnectedness of global economies and the profound impact political events can have on individuals and societies beyond national borders.


**Original Source:** https://www.khq.com/national/economy-and-especially-trump-canadians-thoughts-on-campaigns/article_1628cc95-e4c3-5f09-9f1d-2ddce6de2790.html 


---

## Opinion | Pierre Poilievre’s big problem as Conservatives slide in the polls? He can’t turn his enemies into friends.

**Source:** Toronto Star  
**Publication Time:** 44 minutes ago 


### Summary

This opinion piece in the Toronto Star argues that Conservative leader Pierre Poilievre's relentless attacks on opponents and perceived "enemies" are hindering his ability to build bridges and appeal to a wider electorate. The author suggests that Poilievre's strategy, focused on rallying his base with divisive rhetoric, is alienating moderate voters who could be crucial for the Conservatives to win the next election.

### Key Insights

* Poilievre's campaign strategy relies heavily on framing issues as battles against perceived enemies, such as "elites" and "activists."
* This approach, while effective in mobilizing his base, may be alienating moderate voters who prefer less divisive political discourse.
*  The author suggests that Poilievre needs to broaden his appeal by focusing on unifying themes and building consensus rather than constantly demonizing opponents.

### Broader Implications

This analysis touches on broader societal trends in politics, particularly the rise of polarization and the increasing use of divisive language. It raises questions about the long-term viability of political strategies that rely on demonizing opponents and appealing to a narrow base of support.

### Future Impact

The author suggests that Poilievre's continued reliance on divisive rhetoric could negatively impact the Conservative party's chances in the next election. If he fails to broaden his appeal and connect with moderate voters, the Conservatives may struggle to form a government. 

### Relevance

* **Ethics, Philosophy, or Psychology:** 
    * The article explores the psychological impact of divisive political rhetoric and its potential to alienate voters.
    * It also raises ethical questions about the use of inflammatory language and the responsibility of political leaders to promote civil discourse.

### Conclusion

While Poilievre's strategy of rallying his base with strong rhetoric may be effective in the short term, it may ultimately prove unsustainable in a pluralistic democracy that requires broad-based support for successful governance.


**Original Source:** https://www.thestar.com/politics/political-opinion/pierre-poilievre-s-big-problem-as-conservatives-slide-in-the-polls-he-can-t-turn/article_9049c23c-02b7-4c64-9ab4-9fa7776efe21.html


---

## Poilievre Dangles More Invest-in-Canada Tax Breaks as Vote Nears

**Source:** Bloomberg.com 
**Publication Time:** 16 hours ago

### Summary

Conservative Leader Pierre Poilievre is proposing further tax breaks for investments in Canada as the federal election approaches.  His plan includes an increase in the existing "Invest in Canada" tax credit, which encourages businesses to invest in the country. Poilievre argues that these measures will stimulate economic growth, create jobs, and boost innovation.

### Key Insights

* **Tax Incentives:** The article highlights the Conservative Party's reliance on tax breaks as a key economic strategy. 
* **Investment Focus:** The proposed changes emphasize attracting and incentivizing business investment in Canada. 
* **Election Timing:** The timing of the announcement suggests a deliberate attempt to appeal to voters concerned about economic growth and job creation.

### Broader Implications

* **Competition:** These tax breaks could intensify competition between Canada and other countries vying for foreign investment.
* **Economic Impact:** The effectiveness of tax incentives as a driver of economic growth is a subject of ongoing debate among economists.
* **Social Impact:**  The potential job creation and economic growth touted by Poilievre could have positive social impacts, but it remains to be seen if these benefits will be widely distributed.

### Future Impact

The impact of these proposed tax breaks will depend on several factors, including:

* **Implementation:** The specifics of how the tax credit is increased and its application to different sectors will be crucial.
* **Investment Response:** Whether businesses respond positively to the incentives and increase their investments in Canada remains to be seen.
* **Government Spending:**  The potential revenue loss from these tax breaks could necessitate cuts in other government programs.


### Relevance

* **Economics and Investing:** The article primarily focuses on economic policy, particularly tax incentives and their potential impact on investment.

### Conclusion

Poilievre's proposal to expand the "Invest in Canada" tax credit is a significant economic policy announcement with potential implications for both the Canadian economy and the global investment landscape. The effectiveness and long-term consequences of this policy will depend on various factors and require careful analysis and monitoring.


**Original Source:** https://www.bloomberg.com/news/articles/2025-03-30/poilievre-dangles-more-invest-in-canada-tax-breaks-as-vote-nears

---

## Carney Seen as Best Leader to Represent Canada, Navigate Tough Economic Times

**Source:** Ipsos  **Publication Time:** 15 hours ago

### Summary

A recent Ipsos poll reveals that Mark Carney, former Governor of the Bank of Canada, is widely perceived as the best candidate to lead Canada through challenging economic times.  The survey, conducted in early October 2023, showed Carney garnering 47% support among Canadians when asked who they'd prefer as Prime Minister. This significantly surpasses the support for other potential leaders, including current Prime Minister Justin Trudeau (25%). 

The poll suggests that Canadians place a high value on experience and stability in leadership, particularly during uncertain economic conditions. They view Carney's extensive experience in finance and his track record as Governor of the Bank of Canada as assets that would be beneficial in navigating the current economic climate.

### Key Insights

* **Public Perception of Leadership:** The poll highlights the importance of public perception and trust in a leader, especially during times of economic uncertainty. Canadians seem to prefer a leader with proven experience and a strong track record.
* **Economic Concerns:** The strong support for Carney suggests that economic concerns are a top priority for Canadians. They are looking for a leader who can effectively address issues such as inflation, cost of living, and job security.
* **Shifting Political Landscape:** The poll results indicate a potential shift in public sentiment towards alternative leadership options. The significant gap in support between Carney and Trudeau could signal a desire for change within the Canadian electorate.

### Broader Implications

The poll's findings have broader implications for Canadian politics and the global economic outlook.

* **Conservative Party:** The strong support for Carney, a former Governor of the Bank of Canada, could benefit the Conservative Party, which has traditionally emphasized economic issues. It remains to be seen whether this translates into electoral success.
* **International Confidence:** Carney's perceived competence and experience in navigating economic challenges could potentially boost international confidence in Canada's economic leadership.
* **Global Economic Trends:** The poll reflects a global trend of rising economic anxiety and a desire for stable and experienced leadership.


### Future Impact

The long-term impact of these findings remains to be seen. 

* **Political Motivations:** Political parties may leverage public sentiment towards Carney to shape their strategies and messaging in future elections.
* **Policy Changes:** The public's focus on economic issues could influence government policy decisions and priorities.


### Relevance

* **Economics and Investing:** The article directly relates to economics and investing as it analyzes public perception of economic leadership and potential impact on investment decisions. 
* **Politics:** It also holds relevance to politics, exploring  public sentiment towards potential leaders and the influence of economic concerns on political discourse.

### Conclusion

The Ipsos poll reveals a significant public desire for experienced leadership, particularly in navigating challenging economic times. Mark Carney's strong showing as a preferred candidate highlights the importance of economic concerns in shaping public opinion and the potential for a shift in political landscape.


**Original Source:** https://www.ipsos.com/en-ca/carney-seen-best-leader-represent-canada-navigate-tough-economic-times


---

## Pierre Poilievre Unveils Capital Gains Tax Break Ahead of Election

**Source:** Global News  
**Publication Time:** 15 hours ago

### Summary

Conservative Party Leader Pierre Poilievre announced a plan to reduce the capital gains tax for Canadian small businesses and individuals. The proposal aims to cut the tax rate from 50% to 20%, arguing that it will stimulate economic growth and encourage investment. This announcement comes ahead of the upcoming federal election.  

### Key Insights

* **Tax policy:** The plan focuses on a significant reduction in the capital gains tax, a key area of debate in Canadian politics. 
* **Economic impact:** The Conservative Party claims the tax break will boost economic growth and investment, particularly in small businesses. 
* **Election strategy:** Unveiling this policy ahead of the election suggests an attempt to appeal to voters concerned about taxes and economic prosperity.

### Broader Implications

* **Income inequality:** Critics argue that reducing capital gains taxes disproportionately benefits wealthy individuals and corporations, potentially exacerbating income inequality.
* **Government revenue:** The tax cut could result in a decrease in government revenue, potentially impacting public services and programs.
* **Investment behaviour:** The policy change could influence investment decisions, potentially leading to a shift in capital allocation.

### Future Impact

The long-term impact of this policy depends on various factors, including its implementation, economic conditions, and the outcome of the federal election. 

* **Economic growth:** If the policy stimulates investment and entrepreneurship, it could lead to increased economic growth. However, the extent of this impact remains uncertain.
* **Government finances:** The reduction in government revenue could necessitate spending cuts or tax increases elsewhere, potentially impacting social programs and public services.
* **Political landscape:** The success of this policy could influence future tax debates and shape the Conservative Party's platform in subsequent elections.

### Relevance

* **Economics and Investing:** This article is highly relevant to these domains due to its focus on tax policy, its potential impact on investment behaviour, and the implications for economic growth and government finances.

### Conclusion

Pierre Poilievre's proposal to reduce the capital gains tax is a significant policy announcement with potential implications for the Canadian economy, government finances, and the upcoming federal election. The long-term impact of this policy remains to be seen and will depend on various factors.


**Original Source:** https://globalnews.ca/news/11105333/conservative-party-leader-capital-gains-tax-break/  


---

## Poilievre Shrugs Off Reports of Campaign Turmoil as He Pitches Tax Relief 

**Source:** The Albertan  
**Publication Time:** 11 hours ago 

### Summary

Conservative leader Pierre Poilievre dismissed reports of internal campaign turmoil during a recent speech in Alberta. He focused on his party's tax relief plan, emphasizing its potential to help Canadians struggling with the rising cost of living. Poilievre argued that his plan is fiscally responsible and will stimulate economic growth.

### Key Insights

- Poilievre appears confident in his leadership despite reports of campaign challenges. 
- The Conservative party's campaign strategy centers around economic anxiety and promises of tax relief. 
- Poilievre's message resonates with voters concerned about affordability and the impact of inflation.

### Broader Implications

This event highlights the ongoing political battle over economic policy in Canada. Poilievre's approach aligns with populist rhetoric, appealing to voters' frustrations with the current economic climate.  

### Future Impact

Poilievre's tax relief plan could become a central issue in the next federal election.  Its potential impact on the economy and government finances will likely be debated extensively.

### Relevance

- **Economics and Investing:** Poilievre's tax plan has significant implications for the Canadian economy and investment landscape. 
- **Politics:**  This event provides insight into the internal dynamics of the Conservative party and its campaign strategy.

### Conclusion

Poilievre's continued focus on tax relief suggests it will be a key element of his campaign message. The potential economic and political consequences of his plan remain to be seen.

**Original Source:** https://www.thealbertan.com/politics/poilievre-shrugs-off-reports-of-campaign-turmoil-as-he-pitches-tax-relief-10449528 




---

## Pierre Poilievre Responds to Calls for Conservative Campaign Pivot

**Source:** Yahoo News Canada 
**Publication Time:** 11 hours ago

### Summary

Conservative Party Leader Pierre Poilievre has responded to calls from some within his party to pivot their campaign strategy ahead of the next federal election. Poilievre maintains that the party's current focus on cost of living issues and economic anxieties resonates with Canadians and will remain the central theme of their campaign. He dismisses calls for a more moderate approach, arguing that his party's positions are aligned with the values of the majority of Canadians.

### Key Insights

* Poilievre is confident in his current campaign strategy despite internal pressure for a change.
* The Conservative Party's emphasis on economic issues suggests a focus on appealing to voters concerned about inflation and economic hardship.
* Poilievre's rejection of a more moderate approach indicates a willingness to maintain a strong ideological stance.

### Broader Implications

The internal debate within the Conservative Party reflects a broader tension between populist and moderate wings of the party. Poilievre's stance suggests a continued embrace of populist messaging, which may appeal to some voters but could alienate others. The outcome of this debate could have significant implications for the direction of the Conservative Party and its chances in the next election.

### Future Impact

The long-term impact of Poilievre's strategy remains to be seen. If his focus on economic anxieties resonates with voters, it could solidify his position as leader and boost the Conservative Party's electoral prospects. However, if the public perception shifts or if other parties effectively counter his message, it could have the opposite effect. 

### Relevance

* **Economics and Investing:** The article focuses on economic anxieties and the Conservative Party's stance on cost of living issues, making it relevant to the field of economics. 


### Conclusion

Pierre Poilievre's decision to maintain his current campaign strategy despite calls for a pivot reflects a calculated risk. His focus on economic concerns and his rejection of a moderate approach signal a potential shift in the Conservative Party's direction, with implications for both the upcoming election and the party's future.

**Original Source:** https://ca.news.yahoo.com/pierre-poilievre-responds-calls-conservative-223000321.html 


---

## Pierre Poilievre’s Big Problem as Conservatives Slide in the Polls? He Can’t Turn His Enemies into Friends

**Source:** The Spec  **Publication Time:** 44 minutes ago 

---

### Summary

This opinion piece in The Spec argues that Conservative Leader Pierre Poilievre's strategy of portraying himself as a fighter against "the establishment" is increasingly harming the party's chances in the polls. 

The author contends that while Poilievre's attacks on perceived enemies resonate with some, his inability to build bridges or find common ground with opposing viewpoints is alienating potential voters. This is exemplified by his recent criticisms of the Bank of Canada and Prime Minister Trudeau, which are seen as divisive rather than constructive.

### Key Insights

* Poilievre's "anti-establishment" rhetoric, while effective in galvanizing his base, may be alienating moderate voters.
* The article suggests that Poilievre's focus on attacking opponents rather than proposing solutions hinders his ability to present a compelling alternative to the current government.
* The declining Conservative poll numbers are attributed, in part, to Poilievre's divisive approach.

### Broader Implications

* The article touches on the broader political trend of increasing polarization in Canada, where political discourse often focuses on division rather than collaboration.
* It raises questions about the effectiveness of populism, which often thrives on demonizing opponents, in building a sustainable political movement.

### Future Impact

* If Poilievre continues his current approach, the article suggests it could further damage the Conservative party's electoral prospects.
* The piece implies that a shift towards more constructive and unifying messaging may be necessary for the Conservatives to appeal to a broader range of voters.

### Relevance

* **Politics:** The article directly addresses Canadian politics and the current state of the Conservative party.
* **Social Trends:** It touches on the broader societal trend of political polarization and the impact of populist rhetoric.

### Conclusion

This opinion piece in The Spec argues that Pierre Poilievre's divisive political strategy, while successful in mobilizing his base, may ultimately prove detrimental to the Conservative party's electoral success. His inability to build bridges and engage in constructive dialogue with opposing viewpoints is seen as a significant obstacle to winning over a wider electorate.


**Original Source:** https://www.thespec.com/politics/political-opinion/pierre-poilievre-s-big-problem-as-conservatives-slide-in-the-polls-he-can-t-turn/article_013a65ef-7357-534a-80b2-b42b258c048c.html

---

I can do that. Here is the analysis of the news article you provided:

**Summary**

According to the Barron News-Shield article, liberal Prime Minister Justin Trudeau is currently leading in the polls, four weeks before the Canadian federal election. The article highlights Trudeau's campaign strategy and public approval ratings. It also mentions the Conservative Party leader, Pierre Poilievre, and his stance on the economy, but doesn't delve into his campaign plans in detail. 

**Key Insights**

* **Trudeau's lead is significant:** The article states that Trudeau is enjoying a comfortable lead in the polls, suggesting a potential victory for the Liberal Party.
* **Economy a key issue:**  The article mentions that the economy is a crucial topic in the election campaign, with Poilievre focusing on his economic policies.


**Broader Implications**

* **Political landscape:** Trudeau's potential win could signify a continuation of his liberal policies and agenda for Canada. 
* **Economic policy:**  The outcome of the election will likely have implications for Canada's economic direction, depending on which party forms the government.

**Future Impact**

The election results will shape Canada's political and economic landscape for the next several years.  Trudeau's potential win could lead to continued investments in social programs and climate change initiatives, while a Conservative victory might result in a shift towards more fiscally conservative policies.

**Relevance**

* **Economics and Investing:** The article is relevant to this domain as it discusses the economic policies of the two main parties and how they may impact the Canadian economy.

**Conclusion**

The article paints a picture of a Canadian election where Trudeau is favored to win, with the economy being a key issue. The election results will have significant implications for the future direction of Canada.


**Original Source:** https://www.news-shield.com/news/national/article_eedb3bf4-1c4a-5b12-952f-6dad25afab18.html


---

## Summary

This article reports on a recent poll showing Prime Minister Justin Trudeau and the Liberal Party taking a lead over the Conservative Party four weeks before the Canadian federal election. The poll suggests that 37% of Canadians would vote for the Liberals, compared to 32% for the Conservatives. Other parties, like the NDP and the Bloc Québécois, lag behind with smaller percentages. The article also highlights the main issues driving voter sentiment, including the economy, healthcare, and climate change. 

## Key Insights

* **Electoral Tight Race:** While the Liberals have a slight lead, the race remains close, indicating a potentially tight election.
* **Public Priorities:** The poll suggests that the economy, healthcare, and climate change are top concerns for Canadian voters.
* **Party Platforms:** The article subtly hints at the potential importance of party platforms addressing these key issues in swaying undecided voters.

## Broader Implications

* **Political Landscape:** The poll results provide a snapshot of the evolving political landscape in Canada, reflecting potential shifts in voter preferences. 
* **Policy Focus:** The identified key issues highlight areas where political parties will likely focus their campaign messaging and policy proposals.
* **Public Discourse:** The article reflects public discourse around important national issues and the degree to which they influence electoral choices.

## Future Impact

The outcome of the election will have significant implications for Canada's future direction on issues like the economy, healthcare, and climate change. The Liberal lead, if maintained, suggests a potential continuation of their current policies. However, the close race indicates that the final result remains uncertain and could potentially lead to changes in government priorities.

## Relevance

* **Economics and Investing:**  The article's focus on the economy and its impact on voter sentiment is highly relevant to this domain, as it can influence economic policy and investment decisions.


## Conclusion

The recent poll suggesting a Liberal lead in the Canadian federal election provides valuable insight into the current political climate and public priorities. The outcome of the election will have significant implications for Canada's future direction on key issues facing the nation.

**Original Source:** https://www.wataugademocrat.com/news/national/liberal-pm-carney-takes-lead-four-weeks-before-canada-vote/article_b93e4252-6f15-5556-b4d0-a2ec706b3255.html


---

## Pierre Poilievre touts capital gains tax break in wake of reports of internal party tension

**Source:** www.thestar.com - RSS Results of type article
**Publication Time:** 2025-03-30 23:53:00

### Summary

Conservative Leader Pierre Poilievre used a speech in Calgary to highlight his party's proposed capital gains tax break, framing it as a way to support economic growth and individual prosperity. This announcement comes amidst reports of internal party tension and dissent over Poilievre's leadership style and policy direction.  

### Key Insights

* Poilievre's focus on capital gains tax breaks suggests a strategy to appeal to wealthier voters and promote economic activity through investment.
*  The timing of this announcement, coinciding with reports of internal party conflict, indicates Poilievre may be attempting to consolidate support and project a sense of unity within the Conservative party.

### Broader Implications

*  This policy proposal raises questions about the potential impact on income inequality and the distribution of wealth. 
*  The emphasis on economic growth through individual investment aligns with certain libertarian economic ideologies, potentially fueling debates about the role of government intervention in the economy.

### Future Impact

*  The long-term impact of a capital gains tax break depends on various factors, including the specific details of the policy and the broader economic context.
*  The success of this strategy in addressing internal party divisions and galvanizing support for the Conservative party remains to be seen. 

### Relevance

* **Economics and Investing:** This article is highly relevant to this domain due to its focus on a key economic policy proposal and its potential impact on investment behavior, wealth distribution, and economic growth.

### Conclusion

Pierre Poilievre's announcement of a capital gains tax break is a significant development in Canadian politics. It reflects his economic priorities and his attempt to address internal party challenges. The long-term consequences of this policy proposal remain to be seen, but it is likely to spark debate about its economic and social implications.

**Original Source:** https://www.thestar.com/politics/federal-elections/pierre-poilievre-touts-capital-gains-tax-break-in-wake-of-reports-of-internal-party-tension/article_e0b4175f-b555-4f38-bbbb-be79b044d71f.html

---

##  Is Pierre Poilievre ‘too Trumpy?’ Does Mark Carney know his candidates’ names? Five moments that shook up the election campaign this week 

**Source:** www.thestar.com - RSS Results of type article
**Publication Time:** 2025-03-30 12:00:00 

### Summary

This article from The Star analyzes five key moments that significantly impacted the Canadian federal election campaign during the past week. It delves into Poilievre's potential resemblance to Donald Trump's political style, questions Carney's familiarity with his own party's candidates, and examines the overall political landscape.

### Key Insights

-  Pierre Poilievre's increasingly confrontational and populist rhetoric has drawn comparisons to former U.S. President Donald Trump, raising concerns about its potential impact on Canadian politics.
-  The article highlights an incident where Mark Carney, the leader of the Liberal Party, appeared to struggle to recall the names of some of his own candidates, raising questions about his campaign strategy and leadership.
-  Other notable moments discussed include debates on housing affordability, economic anxieties, and the role of government intervention.

### Broader Implications

The article touches upon broader societal anxieties surrounding political polarization, the influence of American politics on Canadian discourse, and the effectiveness of traditional political leadership in a rapidly changing world.  

### Future Impact

The events discussed in the article could significantly shape the trajectory of the Canadian election campaign. Poilievre's continued embrace of a populist style might galvanize his base but alienate moderate voters.  Carney's perceived lack of connection with his candidates could weaken his campaign and potentially open the door for other contenders.

### Relevance

- **Politics:**  The article is directly relevant to the Canadian political landscape, offering insights into the ongoing election campaign and the key issues at play.
- **Society:** The article explores broader societal trends such as political polarization and the challenges of traditional leadership in a changing world.

### Conclusion

This article provides a timely and insightful analysis of key moments shaping the Canadian election campaign. The events discussed raise important questions about political strategy, leadership, and the future direction of Canadian politics.

**Original Source:** https://www.thestar.com/politics/federal-elections/is-pierre-poilievre-too-trumpy-does-mark-carney-know-his-candidates-names-five-moments-that-shook-up-the-election-campaign-this-week/article_9d2b95fb-1d25-4746-b346-dde0e95a0bbd.html




---

## Summary of  "Pierre Poilievre aligns with Bloc Québécois just as Jagmeet Singh says he ‘will never support’ Conservatives"

Conservative Leader Pierre Poilievre has indicated he's willing to work with the Bloc Québécois on certain issues, despite the Bloc's stance against the Conservatives. This comes after NDP Leader Jagmeet Singh stated his party would never support a Conservative government. The article highlights the evolving dynamics within Canada's Parliament as different parties navigate potential alliances and opposition strategies.

**Key Insights**

* **Political Pragmatism:** Poilievre's willingness to engage with the Bloc Québécois suggests a pragmatic approach to governing and potentially securing support for his agenda.
* **Opposition Strategy:** Singh's firm rejection of Conservative support underscores the NDP's commitment to ideological differences and maintaining a distinct political stance.
* **Shifting Alliances:** The evolving landscape of Canadian politics indicates potential for fluid alliances and shifting power dynamics based on specific issues and policy priorities.

**Broader Implications**

* **Impact on Governance:** The potential for cooperation between the Conservatives and the Bloc Québécois could influence policy outcomes, particularly on issues of importance to Quebec.
* **Rise of Regionalism:** The Bloc Québécois's continued presence and influence highlight the importance of regional interests in Canadian politics.
* **Erosion of Traditional Party Lines:** The willingness to explore alliances beyond traditional party lines could challenge established political norms and create new coalitions.

**Future Impact**

* **Potential for Coalition Governments:** The article suggests a future where minority governments necessitate alliances between parties with differing ideologies.
* **Increased Polarization:** The hardening of stances between the Conservatives and the NDP could lead to further political polarization.
* **Evolving Canadian Identity:** The changing dynamics of Canadian politics may contribute to a more complex and nuanced understanding of national identity.

**Relevance**

This article is primarily relevant to the domain of **Politics**. It also touches upon **Economics and Investing**, as potential policy changes resulting from shifting alliances could impact the Canadian economy.


**Conclusion**

The article highlights a significant moment in Canadian politics, showcasing the evolving dynamics and potential alliances within Parliament. While the future remains uncertain, this shift signifies a departure from traditional political norms and raises important questions about governance, regional interests, and the evolving Canadian identity.

**Original Source:** https://www.thestar.com/politics/federal-elections/pierre-poilievre-aligns-with-bloc-qu-b-cois-just-as-jagmeet-singh-says-he-will-never-support-conservatives/article_7395e3ed-e655-4586-9123-d094c1de818a.html


---

## Pierre Poilievre has the ‘Century Initiative’ and its links to Mark Carney in his crosshairs. So what is it?

**Source:** www.thestar.com - Published: 2025-03-29 14:00:00

### Summary

Conservative Leader Pierre Poilievre is criticizing the "Century Initiative," a project led by former Bank of Canada Governor Mark Carney, accusing it of being a "radical" plan that could undermine Canada's economy. The initiative aims to mobilize trillions of dollars in private capital towards sustainable investment, particularly focusing on climate change mitigation and adaptation. Poilievre argues that the Century Initiative would lead to government overreach and interfere with free market principles. 

### Key Insights

* The Century Initiative represents a potential shift towards a more interventionist approach to tackling climate change, leveraging private capital for sustainable investments.
* Poilievre's criticism highlights the ongoing political debate surrounding climate action, with conservatives emphasizing free market solutions and liberals advocating for government-led initiatives.
* The initiative's focus on mobilizing private capital could have implications for the financial sector and investment strategies.

### Broader Implications

* The debate surrounding the Century Initiative touches upon broader societal discussions about the role of government in addressing global challenges like climate change.
* It raises questions about the balance between economic growth and environmental sustainability, and the potential impact of climate policies on individual liberties and market forces.
* The initiative's success could influence the direction of global climate finance, potentially encouraging other countries to adopt similar approaches.

### Future Impact

* The Century Initiative's long-term impact will depend on its ability to attract significant private investment and deliver tangible results in terms of reducing greenhouse gas emissions.
* Poilievre's criticism could shape the political landscape around climate policy, influencing future debates and potentially hindering the initiative's progress.
* The initiative's success or failure could serve as a model for other countries seeking to leverage private capital for climate action.

### Relevance

* **Economics and Investing:** The initiative's focus on mobilizing private capital for sustainable investments has significant implications for financial markets and investment strategies.
* **Environment and Climate:** The Century Initiative directly addresses the urgent need to mitigate and adapt to climate change, potentially contributing to global efforts to reduce greenhouse gas emissions.


### Conclusion

The "Century Initiative" presents a bold vision for tackling climate change through private sector investment. However, its success hinges on attracting sufficient capital and navigating the complex political landscape.  Poilievre's criticism highlights the ongoing debate surrounding the role of government and market forces in addressing global challenges.


**Original Source:** https://www.thestar.com/politics/federal/pierre-poilievre-has-the-century-initiative-and-its-links-to-mark-carney-in-his-crosshairs-so-what-is-it/article_ee8de8f9-977f-435e-97f7-580a0641ff27.html


In [285]:
response = graph_2.invoke({"question": "omlet"})

In [286]:
response["answer"]

'An omelette is a dish made from beaten eggs that are fried in butter or oil and typically filled with cheese, vegetables, or meat.  \n'

In [287]:
from IPython.display import Markdown
Markdown(response["answer"])

An omelette is a dish made from beaten eggs that are fried in butter or oil and typically filled with cheese, vegetables, or meat.  


# In depth Analysis

In [288]:
analyser = ChatGroq(model="gemma2-9b-it")

In [289]:
news = {'title': 'Philadelphia Phillies vs Washington Nationals: Where and how to watch today’s match, venue, time and more',
   'link': 'https://timesofindia.indiatimes.com/sports/mlb/news/philadelphia-phillies-vs-washington-nationals-where-and-how-to-watch-todays-match-venue-time-and-more/articleshow/119761771.cms',
   'date': '5 hours ago',
   'source': 'Times of India'}

Indepth analyzer

In [290]:
analysis = analyser.invoke(
        [
            SystemMessage(content=analysis_instruction),
            HumanMessage(
                content=f"Here is the news details: {news}"
            ),
        ]
    )

In [291]:
analysis.content

'## Philadelphia Phillies vs Washington Nationals: Where and how to watch today’s match\n\n**Source:** Times of India  \n**Publication Time:** 5 hours ago\n\n### Summary\n\nThis article provides details on how to watch the Philadelphia Phillies vs. Washington Nationals baseball game. It includes the venue (Nationals Park in Washington D.C.), the start time (7:05 PM EDT), and the broadcasting information (ESPN, MLB.TV). \n\n### Key Insights\n\nThe article highlights the specific platforms where fans can access the game, catering to both traditional television viewers and those who prefer streaming.  \n\n### Broader Implications\n\nThe article reflects the growing interest in baseball, particularly in accessing live games through various media channels. \n\n### Future Impact\n\nThe continued availability of live games across multiple platforms is likely to further enhance fan engagement and potentially contribute to the growth of baseball viewership.\n\n### Relevance\n\n* **Sports:** Thi

In [292]:
from IPython.display import Markdown
Markdown(analysis.content)

## Philadelphia Phillies vs Washington Nationals: Where and how to watch today’s match

**Source:** Times of India  
**Publication Time:** 5 hours ago

### Summary

This article provides details on how to watch the Philadelphia Phillies vs. Washington Nationals baseball game. It includes the venue (Nationals Park in Washington D.C.), the start time (7:05 PM EDT), and the broadcasting information (ESPN, MLB.TV). 

### Key Insights

The article highlights the specific platforms where fans can access the game, catering to both traditional television viewers and those who prefer streaming.  

### Broader Implications

The article reflects the growing interest in baseball, particularly in accessing live games through various media channels. 

### Future Impact

The continued availability of live games across multiple platforms is likely to further enhance fan engagement and potentially contribute to the growth of baseball viewership.

### Relevance

* **Sports:** This article is clearly relevant to the sports domain, specifically baseball. 

### Conclusion

The article serves as a convenient guide for fans interested in watching the Phillies vs. Nationals game. 

**Original Source:** https://timesofindia.indiatimes.com/sports/mlb/news/philadelphia-phillies-vs-washington-nationals-where-and-how-to-watch-todays-match-venue-time-and-more/articleshow/119761771.cms
